# Representative Shapes by Embedding Region

For each frame (1, 13), each modality (cell, nucleus), each embedding
method (UMAP, PHATE): k-means clusters that frame's full set of embedded
cells (pooled across all four conditions - `naive`, `vem_1uM`,
`gem_2.5nM`, `tram_6nM`) into a small number of regions, then picks the
one real cell closest to each region's own centroid and shows its
actual shape, cropped from its extracted mask.

Each frame has its own independent embedding (see `01_kde_by_condition.ipynb`
and `config.py`), so this runs once per (frame, modality, embedding
method) - 2 frames x 2 modalities x 2 embedding methods = 8 plots total,
the same structure as `01_kde_by_condition.ipynb`.

Requires `main.py` to have already been run.


In [ ]:
import os
import sys

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/03_multi_drug_comparison/notebooks/), so the analysis dir -
# and config.py - is one level up, and the repo root - where shared/
# lives - is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/MASKS_DIR/CAJAL_OUTPUT_DIR are relative to the
# analysis directory, not this notebook's own cwd - re-anchor them to an
# absolute path once, here (frame_masks_dir/frame_cajal_dir read these
# dynamically, so this must happen before either is ever called)
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.MASKS_DIR        = os.path.join(ANALYSIS_DIR, config.MASKS_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.load_data import load_full_dataset
from shared.representative_shapes import export_representative_shapes


## Plot: embedding regions and their representative shapes

For each (frame, modality, embedding method), one PDF and one SVG (in
its own `svgs/` subfolder), written to `output/plots/00_representative_shapes/`.


In [ ]:
NOTEBOOK_PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '00_representative_shapes')
MODALITY_NAMES = {'cell': 'cell', 'nuc': 'nucleus'}

# how many regions to cluster each frame's embedding into, and how large
# a crop (half-width in pixels) to pull each representative cell's shape
# from - half_width is a guess at "large enough for any cell in this
# dataset"; crop_cell_shape prints a clear warning if a representative
# cell's own shape doesn't fully fit, so bump this up if that happens
N_REGIONS   = 5
HALF_WIDTH  = 150

# main.py computes both of these for every (frame, modality) - plot both
# every time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_frame(frame):
    '''
    For each modality and embedding method, load this frame's full
    embedding (pooled across every condition), cluster it into
    N_REGIONS regions, and export each region's representative shape.

    Inputs:
        frame: frame number (must be one of config.FRAMES)
    Outputs:
        dfs: {(modality, embedding_method): this frame's cells'
              metadata + dim_0/dim_1, plus a 'region' column} (in-memory only)
    '''
    dfs = {}
    for modality in ['cell', 'nuc']:
        for embedding_method in EMBEDDING_METHODS:
            df = load_full_dataset(config.frame_cajal_dir(frame), modality, embedding_method)
            print(f"[frame {frame}/{modality}/{embedding_method}] {len(df)} cells")

            file_prefix = f'{modality}_frame{frame:02d}_{embedding_method}'
            df = export_representative_shapes(
                df, config.frame_masks_dir(frame), modality, N_REGIONS, embedding_method.upper(),
                f'{MODALITY_NAMES[modality]} morphology (representative regions): frame {frame}',
                NOTEBOOK_PLOTS_DIR, file_prefix, half_width=HALF_WIDTH,
            )
            dfs[(modality, embedding_method)] = df

    return dfs


## Run: frame 1

In [ ]:
frame1_dfs = process_frame(config.FRAMES[0])


## Run: frame 13

In [ ]:
frame13_dfs = process_frame(config.FRAMES[1])
